# Hybrid SSA-N-BEATS — Denoising Scenario

Core pipeline only: data split -> SSA decomposition & auto-grouping -> N-BEATS training -> rolling forecast -> evaluation (MAPE, MAE, RMSE, R2).

Reusable logic lives in `src/`; this notebook only orchestrates it.

## Setup

In [ ]:
import os
print(os.getcwd())

d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks


In [2]:
from pathlib import Path
import sys

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [3]:
from pathlib import Path
import os
import sys

print("cwd      :", Path.cwd())
print("resolve  :", Path().resolve())
print("sys.path :", sys.path[:3])  

cwd      : d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks
resolve  : D:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks
sys.path : ['D:\\3. TUGAS AKHIR\\5. TA Alfian\\1_REPO_Q1', 'C:\\Users\\LOQ\\AppData\\Roaming\\uv\\python\\cpython-3.10-windows-x86_64-none\\python310.zip', 'C:\\Users\\LOQ\\AppData\\Roaming\\uv\\python\\cpython-3.10-windows-x86_64-none\\DLLs']


In [4]:
from pathlib import Path

print((Path.cwd() / "../data/desember_cleaned_2024.parquet").resolve())
print((Path.cwd() / "../data/desember_cleaned_2024.parquet").exists())

D:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\data\desember_cleaned_2024.parquet
True


In [1]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.9.0+cpu
None
False
0


## Libs

In [6]:
import warnings
import logging
import json
import numpy as np
import pandas as pd
import torch.nn as nn
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import CSVLogger

from src import (
    SSA,
    compute_w_correlation,
    auto_group_deterministic,
    rolling_forecast_denoising,
    historical_forecast_metrics,
    evaluate_series,
    print_evaluation_report,
)

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data and split (in-sample / out-of-sample)

In [7]:
df = pd.read_parquet("../data/desember_cleaned_2024.parquet")
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])
df = df.sort_values("DATE_TIME").set_index("DATE_TIME")
ts_full = TimeSeries.from_series(df["BEBAN"]).astype(np.float32)

split_date = pd.Timestamp("2024-10-01 00:00:00")
ts_in, ts_out = ts_full.split_before(split_date)
ts_out = ts_out.head(1488)  # 1 month at 30-minute resolution

scaler = Scaler()
ts_in_scaled = scaler.fit_transform(ts_in)
ts_out_scaled = scaler.transform(ts_out)

print(f"In-sample  : {ts_in.start_time()} to {ts_in.end_time()}")
print(f"Out-of-sample (October) : {ts_out.start_time()} to {ts_out.end_time()}")

In-sample  : 2022-01-01 00:00:00 to 2024-09-30 23:30:00
Out-of-sample (October) : 2024-10-01 00:00:00 to 2024-10-31 23:30:00


## 2. SSA decomposition and auto-grouping (Sub-steps 2.1-2.3)

In [8]:
L_window = 336
threshold = 0.9
val_len = 1056

components, s_values = SSA(ts_in_scaled.values().flatten(), window_length=L_window)
w_corr_matrix = compute_w_correlation(components, L_window)
idx_clean = auto_group_deterministic(w_corr_matrix, threshold=threshold)

clean_in_scaled = TimeSeries.from_times_and_values(
    ts_in.time_index, np.sum(components[idx_clean], axis=0)
)

train_clean = clean_in_scaled[:-val_len]
val_clean = clean_in_scaled[-val_len:]

print(f"Selected components (RC): {[i + 1 for i in idx_clean]}")
print(f"Total RC used: {len(idx_clean)} of {L_window}")
print(f"Train length: {len(train_clean)} points")
print(f"Validation length: {len(val_clean)} points")

Selected components (RC): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 14, 15, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 40, 41, 44, 45, 46, 47, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 88, 89, 91, 92, 94, 95, 97, 98, 99, 100, 103, 104, 105, 106, 108, 109, 114, 115, 116, 117, 120, 121, 122, 123, 124, 125, 131, 132, 135, 136, 137, 138, 142, 143, 147, 148, 150, 151, 158, 159, 162, 163, 165, 166, 169, 170, 173, 174, 175, 176, 179, 180, 181, 182, 184, 185, 189, 190, 191, 192, 193, 194, 195, 196, 199, 200, 205, 206, 213, 214, 215, 216, 217, 218, 221, 222, 223, 224, 233, 234, 240, 241, 242, 243, 248, 249, 250, 251, 254, 255, 258, 259, 260, 261, 262, 263, 266, 267, 268, 269, 270, 271, 272, 273, 281, 282, 283, 284, 285, 286, 288, 289, 292, 293, 294, 295, 299, 300, 301, 302, 304, 305, 306, 307, 308, 310, 311, 313, 314, 315, 316, 317, 318, 320, 321, 323, 324, 329, 330, 331, 332, 333, 334, 335, 336]
Total RC used: 206 of 

## 3. Train N-BEATS on the denoised signal (Table 8: October configuration)

In [9]:
with open("../configs/ssa_nbeats_denoising.json") as f:
    cfg = json.load(f)

logger = CSVLogger("logs_ta", name="hybrid_denoise_final_october")

model = NBEATSModel(
    input_chunk_length=cfg["input_chunk_length"],
    output_chunk_length=cfg["output_chunk_length"],
    generic_architecture=True,
    num_stacks=cfg["num_stacks"],
    num_blocks=cfg["num_blocks"],
    num_layers=cfg["num_layers"],
    layer_widths=cfg["layer_widths"],
    dropout=cfg["dropout"],
    batch_size=cfg["batch_size"],
    n_epochs=cfg["n_epochs"],
    optimizer_kwargs={"lr": cfg["learning_rate"]},
    loss_fn=nn.MSELoss(),
    random_state=cfg["random_state"],
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "callbacks": [EarlyStopping(
            monitor="val_loss",
            patience=cfg["early_stopping_patience"],
            min_delta=cfg["min_delta"],
            mode="min",
        )],
        "logger": logger,
        "enable_progress_bar": False,
    },
)

model.fit(series=train_clean, val_series=val_clean, verbose=False)
print("Training finished.")

NameError: name 'exit' is not defined

In [10]:
import pandas as pd
df_check = pd.read_csv(f"{logger.log_dir}/metrics.csv")
print("Total baris log:", len(df_check))
print("Epoch terakhir tercapai:", df_check['epoch'].max())

Total baris log: 712
Epoch terakhir tercapai: 45


## 4. Rolling forecast on the out-of-sample period (Table 4)

In [ ]:
final_denoise_scaled = rolling_forecast_denoising(
    model,
    ts_in_scaled,
    ts_out_scaled,
    window_length=L_window,
    threshold=threshold,
    step_size=48,
)
final_denoise_mw = scaler.inverse_transform(final_denoise_scaled)

## 5. Evaluation (training / validation / out-of-sample)

In [ ]:
ts_in_mw_raw = scaler.inverse_transform(ts_in_scaled)

metrics_train = historical_forecast_metrics(
    model, train_clean, train_clean.time_index[672], scaler, ts_in_mw_raw,
)
metrics_val = historical_forecast_metrics(
    model, clean_in_scaled, val_clean.start_time(), scaler, ts_in_mw_raw,
)
metrics_test = evaluate_series(ts_out, final_denoise_mw)

print_evaluation_report(
    "HYBRID SSA-N-BEATS DENOISING (October 2024)",
    period_labels={
        "training": "Jan 2022 - Aug 2024",
        "validation": "Sep 2024",
        "test": "Oct 2024",
    },
    metrics_by_set={
        "training": metrics_train,
        "validation": metrics_val,
        "test": metrics_test,
    },
)